In [1]:
import json
import pandas as pd
import numpy as np
import yfinance as yf

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Load Data
with open('brokerage_data.json', 'r') as f:
    loaded_data = json.load(f)

sections = {
    'Income': [],
    'Transaction': [],
    'Deposit & Withdrawals': [],
    'Fees': [],
    'Holdings': []
}

for period, data in loaded_data.items():
    for key in sections:        
        if key in data:
            for record in data[key]:
                record['Statement'] = period 
            sections[key].extend(data[key])

trading_data = {key: pd.DataFrame(val) for key, val in sections.items()}

for df in trading_data.values():
    if 'Trade Date' in df.columns:
        df['Trade Date'] = pd.to_datetime(df['Trade Date'])
        df.sort_values(by='Trade Date', inplace=True, ignore_index=True)

# Start & End Date
start_date = pd.concat([
    trading_data['Deposit & Withdrawals']['Trade Date'],
    trading_data['Income']['Trade Date'],
    trading_data['Transaction']['Trade Date']
]).min()

end_date = pd.Timestamp.today()

In [10]:
trading_data['Holdings'].columns

Index(['Symbol', 'Description', 'Quantity', 'Market\nPrice', 'Market\nValue',
       'Cost\nPrice', 'Unrealized', 'TD Cost\nBasis', 'Statement',
       'Market Price', 'Market Value', 'Cost Price', 'TD Cost Basis'],
      dtype='object')

In [ ]:
cashflows = {}
cols = ['Trade Date', 'Net Amt']

# Deposits & Withdrawals
cashflows['Deposits'] = pd.concat(
    [
        trading_data['Deposit & Withdrawals'][cols],
        trading_data['Income'][
            (trading_data['Income']['Entry Type'] == 'Journal Entry(Cash)') &
            (~trading_data['Income']['Description'].str.contains('Deposit to Alpaca Crypto', na=False))
        ][cols]
    ]
).sort_values('Trade Date', ignore_index=True)
cashflows['Deposits'] = cashflows['Deposits'].set_index('Trade Date')

# Dividends + Interest Income + Sale from M&A
cashflows['Net Income'] = pd.concat(
    [
        trading_data['Fees'][cols],
        trading_data['Income'][trading_data['Income']['Entry Type'] != "Journal Entry(Cash)"][cols]
    ]
).sort_values('Trade Date', ignore_index=True)
cashflows['Net Income'] = cashflows['Net Income'].set_index("Trade Date")

In [ ]:
# Benchmark
benchmark = {}
snp500 = yf.Ticker("VOO")
benchmark['VOO'] = snp500.history(start=start_date, end=end_date)
benchmark['VOO'].reset_index(inplace=True)
benchmark['VOO']['Date'] = pd.to_datetime(benchmark['VOO']['Date']).dt.tz_localize(None)
benchmark['VOO'] = benchmark['VOO'].set_index('Date')

date_index = benchmark['VOO'].index

benchmark['VOO performance'] = pd.DataFrame(index=date_index)
benchmark['VOO performance']['Open'] = benchmark['VOO']['Open']
benchmark['VOO performance']['Close'] = benchmark['VOO']['Close']
benchmark['VOO performance']['Deposit'] = 0.0

for dt, amt in cashflows['Deposits']['Net Amt'].items():
    if dt in benchmark['VOO performance'].index:
        benchmark['VOO performance'].at[dt, 'Deposit'] += amt
    else:
        # Optional: insert deposits made on non-trading days to next trading day
        future_idx = benchmark['VOO performance'].index[benchmark['VOO performance'].index > dt]
        if not future_idx.empty:
            benchmark['VOO performance'].at[future_idx[0], 'Deposit'] += amt

benchmark['VOO performance']['Shares Bought'] = benchmark['VOO performance']['Deposit'] / benchmark['VOO performance']['Open']
benchmark['VOO performance']['Shares Bought'] = benchmark['VOO performance']['Shares Bought'].fillna(0)
benchmark['VOO performance']['Total Shares'] = benchmark['VOO performance']['Shares Bought'].cumsum()
benchmark['VOO performance']['Portfolio Value'] = benchmark['VOO performance']['Total Shares'] * benchmark['VOO performance']['Close']
benchmark['VOO performance']['Cumulative Deposits'] = benchmark['VOO performance']['Deposit'].cumsum()

In [ ]:
cumulative_positions = trading_data['Transaction'].groupby('Symbol').sum('Quantity').round(10)
cumulative_positions[cumulative_positions['Quantity'] > 0]

In [ ]:
symbols = np.sort(
    trading_data['Transaction']['Symbol']
    .dropna()
    # [~trading_data['Transaction']['Symbol'].astype(str).str.match(r"^\d")]
    [trading_data['Transaction']['Symbol'] != "219RGT073"]
    .unique()
)

holdings = {}

for name in ['trade', 'holding', 'stock split', 'adj holding', 'price', 'value']:
    holdings[name] = pd.DataFrame(
    0.0,
    index=date_index,
    columns=symbols,
    dtype='float64' 
    )

holdings['trade'].update(
    trading_data['Transaction'].groupby(['Trade Date', 'Symbol'])['Quantity'].sum().unstack(fill_value=0)
)

holdings['holding'] = holdings['trade'].cumsum().round(10)

In [ ]:
def compute_cumulative_split_factors(split_series: pd.Series) -> pd.Series:
    """
    Compute cumulative split factors for retroactive holding adjustment.
    A split on day t affects only days < t.
    """
    factors = split_series.replace(0, 1)
    cumulative = (
        factors[::-1]           # Reverse time
        .ffill()                # Fill backward (i.e., into earlier time)
        .cumprod()              # Cumulative multiply
        [::-1]                  # Flip back to original order
        .shift(-1)              # Shift back so split doesn't apply on the split day
    )
    return cumulative.fillna(1.0)  # If any NaNs remain (e.g., leading NaNs), treat as neutral

In [ ]:
symbols_df = (
    pd.DataFrame({'Symbol': symbols})
    .merge(
        trading_data['Holdings'][['Symbol', 'Description']]
        .drop_duplicates(subset='Symbol'), 
        on='Symbol', how="left")
)

symbols_df['yf_symbol'] = None
symbols_df['yf_name'] = None

holdings['Ticker Info'] = {}

for symbol in symbols:
    try:
        ticker = yf.Ticker(symbol)
        holdings['Ticker Info'][symbol] = {}
        try:
            holdings['Ticker Info'][symbol]['Industry'] = ticker.info['industryDisp']
            holdings['Ticker Info'][symbol]['Sector'] = ticker.info['sectorDisp']
        except:
            holdings['Ticker Info'][symbol]['Holdings'] = "ETF"
        hist = ticker.history(start=start_date, end=end_date)
        hist = hist.reset_index()
        hist['Date'] = pd.to_datetime(hist['Date']).dt.tz_localize(None)
        hist = hist.set_index('Date')

        holdings['price'][symbol] = hist['Close']
        holdings['stock split'][symbol] = hist['Stock Splits']
        holdings['stock split'][symbol] = compute_cumulative_split_factors(holdings['stock split'][symbol])

        holdings['adj holding'][symbol] = holdings['holding'][symbol] * holdings['stock split'][symbol]
        
        symbols_df.loc[symbols_df['Symbol'] == symbol, 'yf_symbol'] = ticker.info.get('symbol')
        symbols_df.loc[symbols_df['Symbol'] == symbol, 'yf_name'] = ticker.info.get('longName')

    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")

holdings['value'] = holdings['adj holding'] * holdings['price']
holdings['Portfolio Value'] = pd.DataFrame(index=date_index)
holdings['Portfolio Value']['Value'] = holdings['value'].sum(axis=1)

In [ ]:
monthly_deposits = cashflows['Deposits']['Net Amt'].resample('ME').sum()
monthly_income = cashflows['Net Income']['Net Amt'].resample('ME').sum()


fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=holdings['Portfolio Value'].index,
        y=holdings['Portfolio Value']['Value'],
        mode='lines',
        name='Personal Portfolio',
        line=dict(color='green', width=2)
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=benchmark['VOO performance'].index,
        y=benchmark['VOO performance']['Portfolio Value'],
        mode='lines',
        name='S&P500 Benchmark',
        line=dict(color='red', width=2)
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=benchmark['VOO performance'].index,
        y=benchmark['VOO performance']['Cumulative Deposits'],
        mode='lines',
        name='Cumulative Net Deposits',
        line=dict(color='darkgrey', width=1)
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Bar(
        x=monthly_deposits.index,
        y=monthly_deposits,
        name='Deposits / Withdrawals',
        marker_color='royalblue',
        opacity=0.5
    ),
    secondary_y=True,
)

fig.update_layout(
    title_text='<b>Portfolio Performance vs. S&P500 Benchmark</b>',
    template='plotly_white',
    barmode='relative',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis=dict(
        title='Date',
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    )
)

fig.update_yaxes(title_text="<b>Portfolio Value ($)</b>", secondary_y=False)
fig.update_yaxes(
    title_text="<b>Monthly Cash Flow ($)</b>",
    secondary_y=True,
    showgrid=False,
    layer='below traces'
)

fig.show()

In [ ]:
fig_income = go.Figure()

fig_income.add_trace(
    go.Bar(
        x=monthly_income.index,
        y=monthly_income,
        name='Dividend / Interest Income',
        marker_color='lightcoral'
    )
)

fig_income.update_layout(
    title_text='<b>Monthly Dividend & Interest Income</b>',
    template='plotly_white',
    xaxis_title='Date',
    yaxis_title='Net Income ($)'
)

fig_income.show()

In [ ]:
trading_data['Holdings']